# Yellow Taxi Analysis: Instructor Solution

Instructor reference, updated for pandas 3. Do not distribute before the debrief.

Differences from the older solution: no `warnings.filterwarnings` (nothing to silence, `SettingWithCopyWarning` no longer exists), and `rename` returns a new DataFrame instead of using `inplace=True`, which reads better under Copy-on-Write.

In [ ]:
from pathlib import Path

import pandas as pd

DATA_PATH = Path("../yellow_tripdata_2026-01.parquet")
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

In [21]:
# read the file
df = pd.read_parquet(DATA_PATH)
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,7.2,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,7.9,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,10.7,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,38.7,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,13.5,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75


In [22]:
# inspect size
df.shape

(3724889, 20)

In [23]:
# inspect the data
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3724889 entries, 0 to 3724888
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     str           
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee            float64   

In [24]:
# calculate Trip Duration in minutes
df["Trip_Duration"] = (
    df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
).dt.total_seconds() // 60

In [25]:
df[['Trip_Duration']].head()

,Trip_Duration
0,5.0
1,5.0
2,8.0
3,42.0
4,13.0


In [26]:
# Total Trip Charge: fare, extra, mta_tax, tolls, improvement surcharge,
# congestion surcharge, airport fee, and tip
df["Total_Trip_Charge"] = (
    df["fare_amount"]
    + df["extra"]
    + df["mta_tax"]
    + df["tolls_amount"]
    + df["improvement_surcharge"]
    + df["congestion_surcharge"]
    + df["Airport_fee"]
    + df["tip_amount"]
)

df[['Total_Trip_Charge']].head()

,Total_Trip_Charge
0,15.86
1,16.15
2,21.45
3,54.81
4,22.35


In [8]:
# most common passenger count
df["passenger_count"].value_counts()

passenger_count
1.0    2150994
2.0     334370
3.0      72864
4.0      49738
0.0      14787
5.0       9184
6.0       4887
8.0          4
7.0          2
9.0          1
Name: count, dtype: int64

In [30]:
# derived date columns
df["Trip_Date"] = pd.to_datetime(df["tpep_pickup_datetime"].dt.date)
df["Trip_Month"] = df["Trip_Date"].dt.month_name()
df["Trip_Day"] = df["Trip_Date"].dt.day_name()
df["Trip_Year"] = df["Trip_Date"].dt.year
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,Trip_Duration,Total_Trip_Charge,Trip_Date,Trip_Month,Trip_Day,Trip_Year
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,...,15.86,2.5,0.0,0.00,5.0,15.86,2026-01-01,January,Thursday,2026
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,...,13.65,2.5,0.0,0.75,5.0,16.15,2026-01-01,January,Thursday,2026
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,...,18.95,2.5,0.0,0.75,8.0,21.45,2026-01-01,January,Thursday,2026
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,...,55.56,2.5,0.0,0.75,42.0,54.81,2026-01-01,January,Thursday,2026
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,...,23.10,2.5,0.0,0.75,13.0,22.35,2026-01-01,January,Thursday,2026


In [31]:
# memory used
df.memory_usage()

Index                         132
VendorID                 14899556
tpep_pickup_datetime     29799112
tpep_dropoff_datetime    29799112
passenger_count          29799112
trip_distance            29799112
RatecodeID               29799112
store_and_fwd_flag       32639411
PULocationID             14899556
DOLocationID             14899556
payment_type             29799112
fare_amount              29799112
extra                    29799112
mta_tax                  29799112
tip_amount               29799112
tolls_amount             29799112
improvement_surcharge    29799112
total_amount             29799112
congestion_surcharge     29799112
Airport_fee              29799112
cbd_congestion_fee       29799112
Trip_Duration            29799112
Total_Trip_Charge        29799112
Trip_Date                29799112
Trip_Month               55873342
Trip_Day                 56803857
Trip_Year                14899556
dtype: int64

In [32]:
# total memory used
total_bytes = df.memory_usage().sum()
print(f"{total_bytes / 1024**2:.2f} MB")   
print(f"{total_bytes / 1024**3:.2f} GB")   

735.38 MB
0.72 GB


In [33]:
# keep and reorder columns
cols = [
    "VendorID", "Trip_Date", "Trip_Year", "Trip_Month", "Trip_Day",
    "passenger_count", "trip_distance", "store_and_fwd_flag",
    "payment_type", "Trip_Duration", "Total_Trip_Charge",
]

df = df[cols]
df.head()

,VendorID,Trip_Date,Trip_Year,Trip_Month,Trip_Day,passenger_count,trip_distance,store_and_fwd_flag,payment_type,Trip_Duration,Total_Trip_Charge
0,2,2026-01-01,2026,January,Thursday,1.0,0.97,N,1,5.0,15.86
1,1,2026-01-01,2026,January,Thursday,0.0,0.90,N,2,5.0,16.15
2,1,2026-01-01,2026,January,Thursday,0.0,1.40,N,1,8.0,21.45
3,2,2026-01-01,2026,January,Thursday,4.0,5.58,N,1,42.0,54.81
4,2,2026-01-01,2026,January,Thursday,0.0,2.16,N,1,13.0,22.35


In [34]:
# rename columns (pandas 3 style: assign the result, no inplace)
df = df.rename(
    columns={
        "VendorID": "Vendor_ID",
        "passenger_count": "No_of_Passengers",
        "store_and_fwd_flag": "SF_Flag",
        "payment_type": "Payment_Type",
    }
)
df.head()

,Vendor_ID,Trip_Date,Trip_Year,Trip_Month,Trip_Day,No_of_Passengers,trip_distance,SF_Flag,Payment_Type,Trip_Duration,Total_Trip_Charge
0,2,2026-01-01,2026,January,Thursday,1.0,0.97,N,1,5.0,15.86
1,1,2026-01-01,2026,January,Thursday,0.0,0.90,N,2,5.0,16.15
2,1,2026-01-01,2026,January,Thursday,0.0,1.40,N,1,8.0,21.45
3,2,2026-01-01,2026,January,Thursday,4.0,5.58,N,1,42.0,54.81
4,2,2026-01-01,2026,January,Thursday,0.0,2.16,N,1,13.0,22.35


In [35]:
# save as parquet partitioned by Year and Month
df.to_parquet(OUT_DIR / "Yellow_Taxi_Transformed", partition_cols=["Trip_Year", "Trip_Month"])

## Challenge 1 Solution

Look at the partition folders that were just created.

In [37]:
for path in sorted((OUT_DIR / "Yellow_Taxi_Transformed").glob("Trip_Year=*")):
    print(path.name)

Strange? Yes. This is the April 2024 file, so there should be exactly one year partition. Instead the partitioning exposed rows from other years. Capture them:

In [ ]:
bad_rows = df[df["Trip_Year"] != 2024]
print("rows outside 2024:", len(bad_rows))
bad_rows[["Trip_Date", "Trip_Year", "Trip_Month", "Total_Trip_Charge"]].sort_values("Trip_Date").head(15)

What happened: TLC monthly files regularly contain a handful of trips whose pickup timestamps fall outside the file's month, from device clock errors and late uploads. Nobody noticed while the data sat in one big file. The moment we partitioned by year, the layout itself surfaced the data quality problem: every bad year became a visible folder.

That is a real production lesson: partitioning is not only a performance tool, it is also a cheap data quality audit. The policy decision (filter to the expected month, quarantine the strays, or keep them with a flag) belongs in your README, exactly like the cleaning policies in Activity 2.

## Challenge 2 Sketch

Combine the remaining 2024 months with the same transformations. The efficient answer is a function applied per month, not copied cells:

In [ ]:
def transform_month(path: Path) -> pd.DataFrame:
    month_df = pd.read_parquet(path)
    month_df["Trip_Duration"] = (
        month_df["tpep_dropoff_datetime"] - month_df["tpep_pickup_datetime"]
    ).dt.total_seconds() // 60
    month_df["Total_Trip_Charge"] = (
        month_df["fare_amount"] + month_df["extra"] + month_df["mta_tax"]
        + month_df["tolls_amount"] + month_df["improvement_surcharge"]
        + month_df["congestion_surcharge"] + month_df["Airport_fee"] + month_df["tip_amount"]
    )
    month_df["Trip_Date"] = pd.to_datetime(month_df["tpep_pickup_datetime"].dt.date)
    month_df["Trip_Month"] = month_df["Trip_Date"].dt.month_name()
    month_df["Trip_Day"] = month_df["Trip_Date"].dt.day_name()
    month_df["Trip_Year"] = month_df["Trip_Date"].dt.year
    keep = [
        "VendorID", "Trip_Date", "Trip_Year", "Trip_Month", "Trip_Day",
        "passenger_count", "trip_distance", "store_and_fwd_flag",
        "payment_type", "Trip_Duration", "Total_Trip_Charge",
    ]
    return month_df[keep].rename(
        columns={
            "VendorID": "Vendor_ID",
            "passenger_count": "No_of_Passengers",
            "store_and_fwd_flag": "SF_Flag",
            "payment_type": "Payment_Type",
        }
    )

# Then, per downloaded month file:
# for path in sorted(Path("data").glob("yellow_tripdata_2024-*.parquet")):
#     transform_month(path).to_parquet(
#         OUT_DIR / "Yellow_Taxi_Transformed",
#         partition_cols=["Trip_Year", "Trip_Month"],
#     )
print("transform_month defined")

Memory note for the debrief: 12 months is roughly 40 million rows. Appending each transformed month into the same partitioned dataset (the loop above) keeps only one month in memory at a time, while `pd.concat` of all months at once may not fit. This question is the perfect bridge to the DataFrame engines bonus lab.